# 📊 diff-integrator Benchmark Results

This notebook provides a comprehensive visual summary of all four diff-integrator benchmarks:

| Benchmark | Protein | Observables | Parameterisation | Epochs |
|---|---|---|---|---|
| **2KZV** | CvR118A (92 res.) | Cα shifts + RDCs (3 media) | Dihedral / NeRF | 500 |
| **GmR58A** | GmR58A (122 res.) | Cα shifts + RDCs (2 media) | Dihedral / NeRF | 500 |
| **HR2876B** | NFU1 N-domain (107 res.) | Cα shifts only | Dihedral / NeRF | 500 |
| **HR2876B Cartesian** | NFU1 N-domain (107 res.) | Cα shifts + bond geometry | **Cartesian** | 894 (auto-stopped / 2000) |

The Cartesian benchmark was added in June 2025 to demonstrate the solution to NeRF
geometric drift on long chains.

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/elkins-lab/diff-integrator/blob/main/examples/interactive_tutorials/visualize_benchmarks.ipynb)

In [ ]:
import sys

if "google.colab" in sys.modules:
    !git clone https://github.com/elkins-lab/diff-integrator.git
    %pip install -q ./diff-integrator matplotlib seaborn numpy py3Dmol
    import os

    os.chdir("diff-integrator/examples/interactive_tutorials")

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import py3Dmol
import seaborn as sns

sns.set_theme(style="whitegrid", font_scale=1.1)
RES = Path("../../benchmarks/results")
COLORS = {
    "2KZV":              "#2ecc71",
    "GmR58A":            "#3498db",
    "HR2876B":           "#e74c3c",
    "HR2876B_cartesian": "#9b59b6",   # purple — Cartesian approach
}

## 📉 Section 1: Optimization Loss Curves

Each benchmark was run with Adam (or Adam + schedule) for a fixed epoch budget.
Key differences:

- **2KZV / GmR58A**: annealed geometry anchor (ExponentialDecaySchedule 10→0.1 / 5→0.1),
  sequence-aware Ramachandran prior (weight=0.5), best-checkpoint by minimum Q-factor.
- **HR2876B (NeRF)**: fixed geometry restraint (weight=1.0), dihedral space, 500 epochs.
- **HR2876B Cartesian**: annealed position anchor (5→0.1), `BondLengthPenalty` (weight=50),
  `BondAnglePenalty` (weight=10), per-term `EarlyStopping` on Cα RMSD (patience=75) —
  run stopped automatically at epoch **894 / 2000** (55% compute saved).

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

for name, color in COLORS.items():
    try:
        h = np.load(RES / name / "loss_history.npy")
        ax1.plot(h, label=f"{name} (final: {h[-1]:.1f})", color=color, linewidth=2)
        # Normalized 0-1
        norm_h = (h - h.min()) / (h.max() - h.min())
        ax2.plot(norm_h, label=name, color=color, linewidth=2)
    except FileNotFoundError:
        print(f"Missing data for {name}")

ax1.set_title("Absolute Loss Value (Log Scale)")
ax1.set_xlabel("Epoch")
ax1.set_ylabel("Total Loss")
ax1.set_yscale("log")
ax1.legend()

ax2.set_title("Normalized Loss (0 to 1)")
ax2.set_xlabel("Epoch")
ax2.set_ylabel("Normalized Loss")
ax2.legend()

plt.tight_layout()
plt.show()

## 🎯 Section 2: Q-factor Results (2KZV)

The Pearson Q-factor measures how well back-calculated RDCs agree with experiment:
$$Q = \sqrt{\frac{\sum_i (D_i^{calc} - D_i^{exp})^2}{\sum_i (D_i^{exp})^2}}$$
Q = 0 is perfect agreement; Q = 1 means the model is no better than random.

Published reference values from **Li et al. (2023)** Table 5 (NMR medoid structure):
- PAG medium: Q = 0.18
- PEG medium: Q = 0.36 *(underdetermined: only 16 RDCs for 5 tensor parameters)*

In [ ]:
try:
    pag_before = np.load(RES / "2KZV/rdc_pag_q_before.npy")
    pag_after = np.load(RES / "2KZV/rdc_pag_q_after.npy")
    peg_before = np.load(RES / "2KZV/rdc_peg_q_before.npy")
    peg_after = np.load(RES / "2KZV/rdc_peg_q_after.npy")

    labels = ["PAG Medium (23 RDCs)", "PEG Medium (16 RDCs)"]
    before_vals = [float(pag_before), float(peg_before)]
    after_vals = [float(pag_after), float(peg_after)]

    x = np.arange(len(labels))
    width = 0.35

    fig, ax = plt.subplots(figsize=(8, 5))
    ax.bar(x - width / 2, before_vals, width, label="Initial (NMR Model 1)", color="#95a5a6")
    ax.bar(x[0] + width / 2, after_vals[0], width, label="Refined (PAG)", color="#2ecc71")
    ax.bar(x[1] + width / 2, after_vals[1], width, label="Refined (PEG)", color="#e74c3c")

    ax.axhline(y=0.18, color="gray", linestyle="--", alpha=0.7)
    ax.text(-0.4, 0.19, "Medoid target (PAG=0.18)", color="gray")

    ax.axhline(y=0.36, color="gray", linestyle="--", alpha=0.7)
    ax.text(0.6, 0.37, "Medoid target (PEG=0.36)", color="gray")

    ax.annotate(
        "⚠️ likely overfitting\n(underdetermined tensor)",
        xy=(1 + width / 2, after_vals[1]),
        xytext=(1.2, 0.2),
        arrowprops={"facecolor": "black", "shrink": 0.05, "width": 1.5, "headwidth": 6},
    )

    ax.set_ylabel("Pearson Q-Factor (lower is better)")
    ax.set_title("2KZV RDC Agreement Before/After Refinement")
    ax.set_xticks(x)
    ax.set_xticklabels(labels)
    ax.legend()

    plt.tight_layout()
    plt.show()
except FileNotFoundError:
    print("2KZV RDC data not found.")

## 🔬 Section 3: Cα Chemical Shift Accuracy

For each protein, we compare predicted Cα chemical shifts (from backbone torsion angles) against the experimentally measured values from the BMRB database.

Prediction method: SPARTA-like soft secondary structure classifier (Gaussian detectors in φ/ψ space) applied to the backbone torsion angles of each structure.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

for ax, name in zip(axes, COLORS.keys(), strict=False):
    try:
        cb = np.load(RES / name / "ca_shifts_before.npy")
        ca = np.load(RES / name / "ca_shifts_after.npy")
        ce = np.load(RES / name / "ca_shifts_exp.npy")

        mask = ~np.isnan(ce)
        rmsd_b = np.sqrt(np.mean((cb[mask] - ce[mask]) ** 2))
        rmsd_a = np.sqrt(np.mean((ca[mask] - ce[mask]) ** 2))

        ax.scatter(
            ce[mask], cb[mask], alpha=0.5, color="gray", label=f"Initial (RMSD={rmsd_b:.2f})"
        )
        ax.scatter(
            ce[mask], ca[mask], alpha=0.7, color=COLORS[name], label=f"Refined (RMSD={rmsd_a:.2f})"
        )

        min_val = min(ce[mask].min(), cb[mask].min(), ca[mask].min())
        max_val = max(ce[mask].max(), cb[mask].max(), ca[mask].max())
        ax.plot([min_val, max_val], [min_val, max_val], "k--", alpha=0.5)

        ax.set_title(f"{name} (N={np.sum(mask)})")
        ax.set_xlabel("Experimental Cα Shift (ppm)")
        ax.set_ylabel("Predicted Cα Shift (ppm)")
        ax.legend()
    except FileNotFoundError:
        ax.set_title(f"{name} - Data Missing")

plt.tight_layout()
plt.show()

## 🏗️ Section 4: NeRF Reconstruction Drift — and the Cartesian Solution

`diff-integrator` originally parameterised backbone conformations as dihedral angles (φ, ψ)
rebuilt via a **NeRF (Natural Extension Reference Frame)** builder using ideal bond geometry.
Over long chains the rounding errors accumulate — called **NeRF drift** — measured as the
Kabsch RMSD between raw PDB Cα positions and the NeRF-rebuilt starting structure.

For HR2876B (107 residues), NeRF drift was ~14 Å, causing the optimizer to start from a
distorted landscape where the geometry anchor competed with the experimental gradient.

### The Cartesian Solution (June 2025)

Starting directly from raw PDB Cα coordinates with `BondLengthPenalty` + `BondAnglePenalty`
completely eliminates reconstruction drift.  The bar chart below shows NeRF drift for the
original three benchmarks.  HR2876B Cartesian has zero drift by construction and is not shown.

| Approach | HR2876B Δ Cα RMSD | Structural drift |
|---|---|---|
| NeRF (dihedral) | −0.011 ppm | 6.4 Å |
| **Cartesian** | **−0.145 ppm (13×)** | **0.24 Å (30× less)** |

In [ ]:
drifts = []
names = []
colors = []
for name, c in COLORS.items():
    if name == "HR2876B_cartesian":
        continue   # Cartesian has zero drift by construction
    try:
        d = float(np.load(RES / name / "nerf_drift.npy"))
        drifts.append(d)
        names.append(name)
        colors.append(c)
    except FileNotFoundError:
        pass

fig, ax = plt.subplots(figsize=(8, 4))
bars = ax.barh(names, drifts, color=colors)

ax.axvline(x=2.0, color="gray", linestyle="--", alpha=0.7)
ax.text(2.1, len(names) - 0.5, "Reliable region (< 2.0 Å)", color="gray")

# Annotation for HR2876B's large drift
for bar, name in zip(bars, names):
    width = bar.get_width()
    label = f"{width:.1f} Å"
    if name == "HR2876B":
        label += "  ← resolved by Cartesian approach"
    ax.text(
        width + 0.1, bar.get_y() + bar.get_height() / 2, label, ha="left", va="center"
    )

ax.set_xlabel("Kabsch RMSD (Å) vs Raw PDB Cα")
ax.set_title("NeRF Reconstruction Drift by Protein\n(HR2876B Cartesian = 0 Å by construction)")
ax.set_xlim(0, max(drifts) + 3 if drifts else 10)
plt.tight_layout()
plt.show()

## 🧬 Section 5: Structural Overlay (2KZV)

The optimized backbone (`final.pdb`) overlaid on the initial NMR model 1 (`initial.pdb`). The backbone drift of ~1.6 Å demonstrates that the geometry anchor is successfully preventing non-physical unravelling while still allowing local improvements.

In [ ]:
try:
    with open(RES / "2KZV/initial.pdb") as f:
        initial_pdb = f.read()
    with open(RES / "2KZV/final.pdb") as f:
        final_pdb = f.read()

    view = py3Dmol.view(width=800, height=500)
    view.addModel(initial_pdb, "pdb")
    view.setStyle({"model": -2}, {"cartoon": {"color": "#e74c3c", "opacity": 0.8}})
    view.addModel(final_pdb, "pdb")
    view.setStyle({"model": -1}, {"cartoon": {"color": "#2ecc71"}})
    view.zoomTo()
    view.show()
except Exception as e:
    print("Could not load 3D viewer. Error:", e)

## 📐 Section 6: Cartesian vs. NeRF Comparison (HR2876B)

The two HR2876B benchmarks run on the same protein and same BMRB dataset, differing only
in their backbone parameterisation.  This direct comparison isolates the effect of
eliminating NeRF drift.

In [ ]:
metrics = ["Δ Cα RMSD (ppm)", "Structural drift (Å)"]
nerf_vals    = [-0.011,  6.356]
cart_vals    = [-0.145,  0.239]

# For display: use absolute values for RMSD improvement, actual values for drift
bar_labels   = ["|Δ Cα RMSD| (ppm, larger=better)", "Structural drift (Å, lower=better)"]
nerf_display = [0.011, 6.356]
cart_display = [0.145, 0.239]

x = np.arange(len(bar_labels))
width = 0.35

fig, ax = plt.subplots(figsize=(10, 5))
bars_n = ax.bar(x - width/2, nerf_display, width, label="HR2876B (NeRF)", color="#e74c3c", alpha=0.8)
bars_c = ax.bar(x + width/2, cart_display, width, label="HR2876B Cartesian", color="#9b59b6", alpha=0.8)

for bar, val in zip(bars_n, nerf_display):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.05,
            f"{val:.3f}", ha="center", va="bottom", fontsize=11)
for bar, val in zip(bars_c, cart_display):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.05,
            f"{val:.3f}", ha="center", va="bottom", fontsize=11, fontweight="bold")

# Improvement callouts
ax.annotate("13× larger\nimprovement", xy=(x[0] + width/2, cart_display[0]),
            xytext=(x[0] + width/2 + 0.3, cart_display[0] + 0.4),
            arrowprops=dict(facecolor="#9b59b6", shrink=0.05, width=1.5, headwidth=6),
            fontsize=10, color="#9b59b6", fontweight="bold")
ax.annotate("30× less drift", xy=(x[1] + width/2, cart_display[1]),
            xytext=(x[1] + width/2 + 0.25, cart_display[1] + 1.0),
            arrowprops=dict(facecolor="#9b59b6", shrink=0.05, width=1.5, headwidth=6),
            fontsize=10, color="#9b59b6", fontweight="bold")

ax.set_xticks(x)
ax.set_xticklabels(bar_labels)
ax.set_title("HR2876B: NeRF vs. Cartesian Parameterisation")
ax.legend()
plt.tight_layout()
plt.show()

print("\nSummary:")
print(f"  NeRF approach  — Δ Cα RMSD: {nerf_vals[0]:+.3f} ppm | structural drift: {nerf_vals[1]:.2f} Å | epochs: 500")
print(f"  Cartesian      — Δ Cα RMSD: {cart_vals[0]:+.3f} ppm | structural drift: {cart_vals[1]:.2f} Å | epochs: 894 (auto-stopped / 2000)")